In [9]:
from pathlib import Path

import pandas as pd
import cmdstanpy
from cmdstanpy import CmdStanModel

cmdstan_path = Path.home() / ".cmdstan" / "cmdstan-2.38.0"
if cmdstan_path.exists():
    cmdstanpy.set_cmdstan_path(str(cmdstan_path))


In [10]:
data = pd.read_csv("data/response_times.csv", sep=";")

data_dict = {
    "N": int(len(data)),
    "J": int(data["id"].nunique()),
    "id": data["id"].astype(int).tolist(),
    "y": data["rt"].astype(float).tolist(),
    "choice": data["choice"].astype(int).tolist(),
    "condition": data["condition"].astype(int).tolist(),
}

data.head()


,id,rt,choice,condition
0,1,1.4456,1,1
1,1,1.7426,1,1
2,1,2.9506,1,1
3,1,1.0676,1,1
4,1,4.9116,1,1


In [11]:

model = CmdStanModel(stan_file="models/diffusion_multiple.stan")


17:59:15 - cmdstanpy - INFO - compiling stan file C:\Users\Victor\Documents\School\CognitiveModeling_TestRepo\hw3\problem6\models\diffusion_multiple.stan to exe file C:\Users\Victor\Documents\School\CognitiveModeling_TestRepo\hw3\problem6\models\diffusion_multiple.exe
17:59:24 - cmdstanpy - INFO - compiled model executable: C:\Users\Victor\Documents\School\CognitiveModeling_TestRepo\hw3\problem6\models\diffusion_multiple.exe


In [12]:
fit = model.sample(
    data=data_dict,
    chains=4,
    parallel_chains=4,
    iter_warmup=500,
    iter_sampling=1000,
    seed=123,
)

summary = fit.summary()


17:59:32 - cmdstanpy - INFO - CmdStan start processing
chain 1:   0%|          | 0/1500 [00:00<?, ?it/s, (Warmup)]





chain 1:   7%|▋         | 100/1500 [00:01<00:26, 53.84it/s, (Warmup)]


chain 1:  13%|█▎        | 200/1500 [00:02<00:16, 77.03it/s, (Warmup)]


chain 1:  20%|██        | 300/1500 [00:03<00:12, 95.16it/s, (Warmup)]


chain 1:  27%|██▋       | 400/1500 [00:04<00:09, 111.17it/s, (Warmup)]





chain 1:  33%|███▎      | 500/1500 [00:05<00:08, 113.55it/s, (Sampling)]


chain 1:  40%|████      | 600/1500 [00:05<00:07, 124.50it/s, (Sampling)]


chain 1:  47%|████▋     | 700/1500 [00:06<00:06, 131.38it/s, (Sampling)]


chain 1:  53%|█████▎    | 800/1500 [00:07<00:05, 137.61it/s, (Sampling)]


chain 1:  60%|██████    | 900/1500 [00:07<00:04, 140.55it/s, (Sampling)]


chain 1:  67%|██████▋   | 1000/1500 [00:08<00:03, 142.27it/s, (Sampling)]


chain 1:  73%|███████▎  | 1100/1500 [00:09<00:02, 143.16it/s, (Sampling)]


chain 1:  80%|████████  | 1200/1500 [00:09<00:02, 144.96it/s,


17:59:44 - cmdstanpy - INFO - CmdStan done processing.
17:59:44 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: wiener_lpdf: Boundary separation is 0, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
	Exception: wiener_lpdf: Boundary separation is 0, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
	Exception: wiener_lpdf: Boundary separation is inf, but must be positive finite! (in 'diffusion_multiple.stan', line 68, column 16 to line 73, column 18)
	Exception: wiener_lpdf: Boundary separation is 0, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
	Exception: wiener_lpdf: Boundary separation is 0, but must be positive finite! (in 'diffusion_multiple.stan', line 61, column 16 to line 66, column 18)
Exception: wiener_lpdf: Boundary separation is inf, but must be positive finite! (in 'diffusion_multiple.stan', 

In [ ]:
drift_summary = pd.DataFrame({
    "operator": list(range(1, len(data)+1)),
    "v_condition_1": [summary.loc[f"v1[{j}]", "Mean"] for j in range(1, len(data)+1)],
    "v_condition_2": [summary.loc[f"v2[{j}]", "Mean"] for j in range(1, len(data)+1)],
})

drift_summary["harder_condition"] = [
    1 if row.v_condition_1 < row.v_condition_2 else 2
    for row in drift_summary.itertuples()
]

drift_summary.round(3)


,operator,v_condition_1,v_condition_2,harder_condition
0,1,0.828,3.559,1
1,2,0.731,3.822,1
2,3,0.670,3.440,1
3,4,0.477,3.387,1
4,5,0.258,3.451,1


In [ ]:
avg_v1 = drift_summary["v_condition_1"].mean()
avg_v2 = drift_summary["v_condition_2"].mean()

print(f"Average drift for condition 1: {avg_v1:.3f}")
print(f"Average drift for condition 2: {avg_v2:.3f}")

if avg_v1 < avg_v2:
    print("Condition 1 is the difficult field.")
else:
    print("Condition 2 is the difficult field.")


Task difficulty is best reflected by the drift rate v.
Average drift for condition 1: 0.593
Average drift for condition 2: 3.532
Condition 1 corresponds to the high-interference (difficult) field.


In [16]:
print(f"Max R-hat: {summary['R_hat'].max():.4f}")
print(f"Min bulk ESS: {summary['ESS_bulk'].min():.1f}")
print(f"Min tail ESS: {summary['ESS_tail'].min():.1f}")

print()
print(fit.diagnose())


Max R-hat: 1.0024
Min bulk ESS: 1664.6
Min tail ESS: 2159.6

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
No divergent transitions found.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete, no problems detected.

